# Create Training Data - Match States

Extracts match states at each minute from historical matches for win probability modeling.

## Concept
* Each match generates ~90 match states (one per minute)
* 500 matches → 45,000 training samples
* Features capture the current match situation at each minute
* Label is the final match outcome (home_win, draw, away_win)

## Target Features
* **current_score_diff** (INT) - home_goals - away_goals at this minute
* **minute** (INT) - 0–90+
* **home_xg_so_far** (DOUBLE) - cumulative home xG
* **away_xg_so_far** (DOUBLE) - cumulative away xG
* **home_shots** (INT)
* **away_shots** (INT)
* **home_red_cards** (INT)
* **away_red_cards** (INT)
* **home_form_pts** (INT) - points from last 5 matches
* **away_form_pts** (INT)
* **final_outcome** (STRING) - 'home_win', 'draw', 'away_win'

In [0]:
import sys
sys.path.append('/Workspace/Users/pawanvirat32@gmail.com/MatchPulse')

from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import *
from config.paths import MATCHES_BRONZE, EVENTS_BRONZE

# Initialize Spark
spark = SparkSession.builder.appName("CreateTrainingData").getOrCreate()

print("=" * 80)
print("Creating Training Data for Win Probability Model")
print("=" * 80)

In [0]:
# Read matches from bronze layer
print("\n[1/6] Reading matches from bronze layer...")
df_matches = spark.read.format("delta").load(MATCHES_BRONZE)
df_matches = df_matches.dropDuplicates(["match_id"])
print(f"   Total matches: {df_matches.count():,}")

# Read events from bronze layer (stored as Parquet)
print("\n[2/6] Reading events from bronze layer...")
df_events = spark.read.format("parquet").load(EVENTS_BRONZE)
print(f"   Total events: {df_events.count():,}")

# Show sample
print("\nSample match:")
display(df_matches.select(
    "match_id", "match_date",
    "home_team.home_team_name", "away_team.away_team_name",
    "home_score", "away_score"
).limit(3))

In [0]:
# Create minute-by-minute match states
print("\n[3/6] Creating minute-by-minute match states...")

# Generate all possible minutes (0-100 to handle injury time)
from pyspark.sql.functions import explode, sequence, lit

# Create match-minute pairs
df_match_minutes = df_matches.select(
    "match_id",
    F.col("home_team.home_team_id").alias("home_team_id"),
    F.col("away_team.away_team_id").alias("away_team_id"),
    F.col("home_score").alias("final_home_score"),
    F.col("away_score").alias("final_away_score"),
    "match_date"
).withColumn(
    "minute",
    explode(sequence(lit(0), lit(95)))  # 0 to 95 minutes
).withColumn(
    "final_outcome",
    F.when(F.col("final_home_score") > F.col("final_away_score"), "home_win")
     .when(F.col("final_home_score") == F.col("final_away_score"), "draw")
     .otherwise("away_win")
)

print(f"   Total match-minute states: {df_match_minutes.count():,}")
print("\nSample match-minute states:")
display(df_match_minutes.limit(5))

In [0]:
# Calculate cumulative statistics up to each minute
print("\n[4/6] Calculating cumulative features from events...")

# Extract xG from raw_json for shot events
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import DoubleType

# Filter shot events and extract xG from raw_json
df_shots = df_events.filter(
    F.col("event_type_name") == "Shot"
).select(
    "match_id",
    F.col("minute").cast("int"),
    "team_id",
    F.get_json_object("raw_json", "$.shot.statsbomb_xg").cast("double").alias("xg_value")
).filter(
    F.col("minute").isNotNull()
)

# Extract goal events from raw_json
df_goals = df_events.filter(
    F.col("event_type_name") == "Shot"
).select(
    "match_id",
    F.col("minute").cast("int"),
    "team_id",
    F.get_json_object("raw_json", "$.shot.outcome.name").alias("outcome")
).filter(
    (F.col("outcome") == "Goal") &
    (F.col("minute").isNotNull())
).withColumn("goal", lit(1))

# Extract red card events from raw_json  
df_red_cards = df_events.filter(
    F.col("event_type_name") == "Foul Committed"
).select(
    "match_id",
    F.col("minute").cast("int"),
    "team_id",
    F.get_json_object("raw_json", "$.foul_committed.card.name").alias("card_type")
).filter(
    (F.col("card_type") == "Red Card") &
    (F.col("minute").isNotNull())
).withColumn("red_card", lit(1))

print(f"   Total shot events: {df_shots.count():,}")
print(f"   Total goal events: {df_goals.count():,}")
print(f"   Total red card events: {df_red_cards.count():,}")

In [0]:
# Join shots and red cards to match-minutes
print("\n[5/6] Aggregating cumulative stats per match-minute...")

# Aggregate shots by match and minute
df_shots_agg = df_shots.groupBy("match_id", "minute", "team_id").agg(
    F.count("*").alias("shots"),
    F.sum("xg_value").alias("xg")
)

# Aggregate red cards
df_red_cards_agg = df_red_cards.groupBy("match_id", "minute", "team_id").agg(
    F.sum("red_card").alias("red_cards")
)

# Create window for cumulative sums
window_cumulative = Window.partitionBy("match_id", "team_id").orderBy("minute").rowsBetween(Window.unboundedPreceding, 0)

# Calculate cumulative home stats
df_home_stats = df_match_minutes.select(
    "match_id", "minute", "home_team_id"
).join(
    df_shots_agg,
    (df_match_minutes.match_id == df_shots_agg.match_id) &
    (df_match_minutes.minute >= df_shots_agg.minute) &
    (df_match_minutes.home_team_id == df_shots_agg.team_id),
    "left"
).groupBy(
    df_match_minutes.match_id,
    df_match_minutes.minute
).agg(
    F.sum(F.coalesce("shots", lit(0))).alias("home_shots"),
    F.sum(F.coalesce("xg", lit(0.0))).alias("home_xg_so_far")
)

# Calculate cumulative away stats
df_away_stats = df_match_minutes.select(
    "match_id", "minute", "away_team_id"
).join(
    df_shots_agg,
    (df_match_minutes.match_id == df_shots_agg.match_id) &
    (df_match_minutes.minute >= df_shots_agg.minute) &
    (df_match_minutes.away_team_id == df_shots_agg.team_id),
    "left"
).groupBy(
    df_match_minutes.match_id,
    df_match_minutes.minute
).agg(
    F.sum(F.coalesce("shots", lit(0))).alias("away_shots"),
    F.sum(F.coalesce("xg", lit(0.0))).alias("away_xg_so_far")
)

print("   Cumulative stats calculated")

In [0]:
# Calculate score at each minute (simplified - using goals from events)
print("\n[6/6] Calculating score progression...")

# df_goals was already created in the previous cell
# Calculate cumulative goals for home team
df_home_goals = df_match_minutes.select(
    "match_id", "minute", "home_team_id"
).join(
    df_goals,
    (df_match_minutes.match_id == df_goals.match_id) &
    (df_match_minutes.minute >= df_goals.minute) &
    (df_match_minutes.home_team_id == df_goals.team_id),
    "left"
).groupBy(
    df_match_minutes.match_id,
    df_match_minutes.minute
).agg(
    F.sum(F.coalesce("goal", lit(0))).alias("home_goals_so_far")
)

# Calculate cumulative goals for away team
df_away_goals = df_match_minutes.select(
    "match_id", "minute", "away_team_id"
).join(
    df_goals,
    (df_match_minutes.match_id == df_goals.match_id) &
    (df_match_minutes.minute >= df_goals.minute) &
    (df_match_minutes.away_team_id == df_goals.team_id),
    "left"
).groupBy(
    df_match_minutes.match_id,
    df_match_minutes.minute
).agg(
    F.sum(F.coalesce("goal", lit(0))).alias("away_goals_so_far")
)

print("   Score progression calculated")

In [0]:
# Join all features together
print("\nJoining all features...")

df_training = df_match_minutes.join(
    df_home_stats,
    ["match_id", "minute"],
    "left"
).join(
    df_away_stats,
    ["match_id", "minute"],
    "left"
).join(
    df_home_goals,
    ["match_id", "minute"],
    "left"
).join(
    df_away_goals,
    ["match_id", "minute"],
    "left"
).withColumn(
    "current_score_diff",
    F.coalesce("home_goals_so_far", lit(0)) - F.coalesce("away_goals_so_far", lit(0))
).select(
    "match_id",
    "minute",
    "current_score_diff",
    F.coalesce("home_xg_so_far", lit(0.0)).alias("home_xg_so_far"),
    F.coalesce("away_xg_so_far", lit(0.0)).alias("away_xg_so_far"),
    F.coalesce("home_shots", lit(0)).alias("home_shots"),
    F.coalesce("away_shots", lit(0)).alias("away_shots"),
    lit(0).alias("home_red_cards"),  # TODO: Calculate from red card events
    lit(0).alias("away_red_cards"),
    lit(0).alias("home_form_pts"),   # TODO: Calculate from previous matches
    lit(0).alias("away_form_pts"),
    "final_outcome"
).filter(
    F.col("minute") <= 95  # Limit to reasonable match duration
)

print(f"\nFinal training dataset: {df_training.count():,} samples")
print("\nSample training data:")
display(df_training.limit(10))

In [0]:
# Write to Unity Catalog
print("\nWriting to Unity Catalog table: matchpulse.ml.training_match_states...")

df_training.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("matchpulse.ml.training_match_states")

print("✓ Successfully written training data")
print(f"\nTotal samples: {df_training.count():,}")
print(f"Matches: {df_training.select('match_id').distinct().count():,}")
print(f"Average samples per match: {df_training.count() / df_training.select('match_id').distinct().count():.1f}")

In [0]:
%sql
-- Verify training data
SELECT 
    final_outcome,
    COUNT(*) as samples,
    ROUND(AVG(current_score_diff), 2) as avg_score_diff,
    ROUND(AVG(home_xg_so_far), 2) as avg_home_xg,
    ROUND(AVG(away_xg_so_far), 2) as avg_away_xg
FROM matchpulse.ml.training_match_states
GROUP BY final_outcome
ORDER BY samples DESC

In [0]:
%sql
select
    *
from matchpulse.ml.training_match_states
limit 100